In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

results_dir = Path(
        "/home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/results/fiben_mongo_sf10"
)

agg = pd.read_csv(results_dir / "benchmark_aggregate_results.csv")

# ============================================================
# QUERY ID
# ============================================================

agg["official_id"] = agg["query_name"].str.extract(r"^(Q\d+)")

# ============================================================
# QUERY GROUP (MESMO PADRÃO NOVO)
# ============================================================

def query_group(qid):
    if qid in ["Q1", "Q2"]:
        return "lookup"
    if qid in ["Q3", "Q4", "Q5", "Q6"]:
        return "complex_read"
    if qid in ["Q7", "Q8", "Q9"]:
        return "aggregation"
    if qid == "Q10":
        return "insert"
    return "other"

agg["query_group"] = agg["official_id"].apply(query_group)

# ============================================================
# HOT ONLY
# ============================================================

hot = agg[agg["run_phase"] == "hot"].copy()

# ============================================================
# MAIN ANALYSIS (NOVO PADRÃO)
# ============================================================

rows = []

n_C = 10
NEAR_THRESHOLD = 0.05  # 5%

for query_name, grp in hot.groupby("query_name"):

    grp = grp.sort_values("p95_latency_ms")

    best_all = grp.iloc[0]
    best_p95 = best_all["p95_latency_ms"]

    # ---------------------------
    # NEAR-BEST (5%)
    # ---------------------------
    grp["near_best"] = (
        (grp["p95_latency_ms"] - best_p95) / best_p95
    ) <= NEAR_THRESHOLD

    # ---------------------------
    # ACTIVATED / PRIMARY
    # ---------------------------
    activated = grp[grp["final_benchmark_group"] != "control"]
    primary = grp[grp["final_benchmark_group"] == "primary"]

    best_activated = activated.loc[activated["p95_latency_ms"].idxmin()]

    # ---------------------------
    # PRIMARY
    # ---------------------------
    if len(primary) > 0:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        primary_regret = (
            best_primary["p95_latency_ms"] - best_p95
        ) / best_p95
    else:
        best_primary = None
        primary_regret = np.nan

    # ---------------------------
    # DSR
    # ---------------------------
    n_A = activated["candidate_id"].nunique()
    dsr = 1 - (n_A / n_C)

    # ---------------------------
    # BUILD ROW
    # ---------------------------
    rows.append({
        "official_id": best_all["official_id"],
        "query_name": query_name,
        "query_group": best_all["query_group"],

        "n_tested_configs": grp["candidate_id"].nunique(),
        "n_activated_configs": n_A,
        "DSR": dsr,

        "best_config": best_all["g_class"],
        "best_group": best_all["final_benchmark_group"],
        "best_design_pattern": best_all["design_pattern"],
        "best_p95_ms": best_p95,

        "top1_preserved_by_activated": (
            best_activated["candidate_id"] == best_all["candidate_id"]
        ),

        "activated_regret": (
            best_activated["p95_latency_ms"] - best_p95
        ) / best_p95,

        "best_primary_config": None if best_primary is None else best_primary["g_class"],
        "best_primary_p95_ms": None if best_primary is None else best_primary["p95_latency_ms"],
        "primary_regret": primary_regret,

        # 🔥 NOVO (IMPORTANTE)
        "n_near_best": grp["near_best"].sum(),
        "near_best_ratio": grp["near_best"].mean(),
    })

analysis_df = pd.DataFrame(rows).sort_values("official_id")

display(analysis_df)

# ============================================================
# GLOBAL METRICS
# ============================================================

print("\n=== GLOBAL METRICS ===")
print("Average DSR:", analysis_df["DSR"].mean())
print("Top-1 preservation activated:", analysis_df["top1_preserved_by_activated"].mean())
print("Mean activated regret:", analysis_df["activated_regret"].mean())
print("Mean primary regret:", analysis_df["primary_regret"].dropna().mean())
print("Mean near-best ratio:", analysis_df["near_best_ratio"].mean())

# ============================================================
# SAVE (PADRÃO FINAL)
# ============================================================

output_path = results_dir / "schemalens_reduction_analysis_hot.csv"
analysis_df.to_csv(output_path, index=False)

print(f"\nSaved to: {output_path}")

,official_id,query_name,query_group,n_tested_configs,n_activated_configs,DSR,best_config,best_group,best_design_pattern,best_p95_ms,top1_preserved_by_activated,activated_regret,best_primary_config,best_primary_p95_ms,primary_regret,n_near_best,near_best_ratio
1,Q1,Q1_CompanyProfileIBM,lookup,2,1,0.9,CONTROL,control,normalized_reference_baseline,0.295784,False,0.155697,G0,0.341837,0.155697,1,0.500000
0,Q10,Q10_CreateAccountHoldingAndBuyTransaction,insert,7,6,0.4,G3,primary,association_references,0.427061,True,0.000000,G3,0.427061,0.000000,2,0.285714
2,Q2,Q2_CompanyWithIndustryCountryAndListedSecurities,lookup,6,5,0.5,G1,primary,embedded_descriptors,0.155368,True,0.000000,G1,0.155368,0.000000,1,0.166667
3,Q3,Q3_SecuritiesHeldInEachFinancialServiceAccount,complex_read,7,6,0.4,G4,primary,deep_nested_document,0.899143,True,0.000000,G4,0.899143,0.000000,2,0.285714
4,Q4,Q4_CompaniesReachedFromPersonThroughAccountHol...,complex_read,7,6,0.4,G5,primary,shared_target_reference_strategy,4.308413,True,0.000000,G5,4.308413,0.000000,3,0.428571
5,Q5,Q5_ReportsAndMetricDataOfCompany,complex_read,4,3,0.7,G9,secondary_affected,benchmark_tradeoff_alternative,28.432783,True,0.000000,G2,47.676557,0.676816,1,0.250000
6,Q6,Q6_TechUSListedSecuritiesWithHighLastTradedValue,complex_read,6,5,0.5,CONTROL,control,normalized_reference_baseline,91.474332,False,0.671524,G5,152.901578,0.671524,1,0.166667
7,Q7,Q7_PersonsWhoBoughtMoreIBMThanSold,aggregation,8,7,0.3,G6,primary,aggregation_materialized_collection,0.857698,True,0.000000,G6,0.857698,0.000000,2,0.250000
8,Q8,Q8_IBMTransactionsBelowAverageSellingPrice,aggregation,6,5,0.5,G5,primary,shared_target_reference_strategy,0.921652,True,0.000000,G5,0.921652,0.000000,2,0.333333
9,Q9,Q9_PersonsWhoBoughtAndSoldSameStock,aggregation,7,6,0.4,G5,primary,shared_target_reference_strategy,1.056870,True,0.000000,G5,1.056870,0.000000,5,0.714286



=== GLOBAL METRICS ===
Average DSR: 0.5000000000000001
Top-1 preservation activated: 0.8
Mean activated regret: 0.08272213251334984
Mean primary regret: 0.15040377751316905
Mean near-best ratio: 0.3380952380952381

Saved to: /home/jovyan/privado/framework evaluation approachs/framework with dataset fiben/results/fiben_mongo_sf10/schemalens_reduction_analysis_hot.csv
